
# March Madness Bracket Optimizer

This notebook consolidates your earlier supervised and unsupervised experiments into one clean, tournament-focused workflow.

It is designed to help you maximize **expected bracket points**, not just game-level accuracy.

## What this notebook does
1. Loads historical game data and standardizes column names
2. Engineers matchup-difference features
3. Trains multiple probabilistic models
4. Compares models using **log loss**, **Brier score**, **ROC AUC**, and **accuracy**
5. Builds an ensemble probability model
6. Optionally explores unsupervised clusters/archetypes
7. Simulates the tournament thousands of times
8. Produces:
   - team round advancement probabilities
   - title probabilities
   - an optional expected-value view using public pick percentages
   - a suggested bracket pick file from a bracket matchup table

## Expected input files
### Historical training data CSV
A row-per-game dataset with columns like:
- team ratings for both teams, such as `Away OR`, `Away DR`, `Home OR`, `Home DR`
- optional seed columns
- game outcome target such as `Away Win`, `Team1_Win`, `Home Win`, etc.

### Bracket matchup CSV
A row-per-game tournament bracket file, one row per potential game. At minimum it should contain:
- `Game_ID`
- `Round`
- `Region`
- `Slot`
- `Team1`
- `Team2`

Optional columns:
- `Seed1`, `Seed2`
- public pick percentages such as `PickPct1`, `PickPct2`

### Team stats CSV
A team-level table used to score new matchups. At minimum it should contain:
- `Team`
- rating/stat columns that match the training features

You can change all file paths in the config cell below.


In [ ]:

# =========================
# 0. Optional installs
# =========================
# Uncomment if needed
# %pip install pandas numpy scikit-learn matplotlib


In [11]:
# =========================
# 1. Imports and config
# =========================
from __future__ import annotations

import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_predict, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# ---------- File paths ----------
TRAIN_PATH = Path(r"c:\Users\kevin\OneDrive\Desktop\March Madness Model\Mar 9 GTDCleaned_Trimmed.csv")       # <- change to your training file
TEAM_STATS_PATH = Path(r"c:\Users\kevin\OneDrive\Desktop\March Madness Model\team_stats.csv")        # <- change to current-season team stats
BRACKET_PATH = Path(r"c:\Users\kevin\OneDrive\Desktop\March Madness Model\bracket_matchups.csv")     # <- change to the tournament bracket file

# ---------- Output paths ----------
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# ---------- Simulation settings ----------
N_SIMS = 50000
RANDOM_SEED = 42

# ---------- Bracket scoring ----------
# Keep this configurable in case scoring format changes.
ROUND_POINTS = {
    1: 10,
    2: 20,
    3: 40,
    4: 80,
    5: 160,
    6: 320,
}

# ---------- Model settings ----------
CV = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [12]:

# =========================
# 2. Column utilities
# =========================
def normalize_cols(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]
    return out

def first_existing(df: pd.DataFrame, candidates: list[str], required: bool = True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"Could not find any of these columns: {candidates}")
    return None

def get_target_series(df: pd.DataFrame) -> tuple[pd.Series, str]:
    candidates = [
        "Away Win", "AwayWin", "away_win",
        "Team1_Win", "Team1 Win", "team1_win",
        "Target", "target",
        "WinnerFlag", "winner_flag",
        "Home Win", "HomeWin", "home_win",
    ]
    target_col = first_existing(df, candidates, required=True)
    y = df[target_col].copy()

    # Convert common encodings to 0/1
    if y.dtype == object:
        y = y.astype(str).str.strip().str.lower().map({
            "1": 1, "0": 0,
            "true": 1, "false": 0,
            "yes": 1, "no": 0,
            "win": 1, "loss": 0,
            "away": 1, "home": 0,
            "team1": 1, "team2": 0,
        })
    y = pd.to_numeric(y, errors="coerce")
    if set(pd.Series(y).dropna().unique()) - {0, 1}:
        raise ValueError(f"Target column '{target_col}' must be binary after cleaning.")
    return y.astype("Int64"), target_col

def detect_side_columns(df: pd.DataFrame) -> dict[str, tuple[str|None, str|None]]:
    # Canonical feature families and common aliases
    families = {
        "off_eff": ["Away OR", "Team1 OR", "A_OR", "Away_Off", "Away AdjO", "Away Adj OE", "Away AdjOE",
                    "Home OR", "Team2 OR", "H_OR", "Home_Off", "Home AdjO", "Home Adj OE", "Home AdjOE"],
        "def_eff": ["Away DR", "Team1 DR", "A_DR", "Away_Def", "Away AdjD",
                    "Home DR", "Team2 DR", "H_DR", "Home_Def", "Home AdjD"],
        "tempo":   ["Away AT", "Team1 AT", "A_AT", "Away_Tempo", "Away AdjT", "Away Pace",
                    "Home AT", "Team2 AT", "H_AT", "Home_Tempo", "Home AdjT", "Home Pace"],
        "seed":    ["Away Seed", "Seed1", "Team1 Seed", "A_Seed",
                    "Home Seed", "Seed2", "Team2 Seed", "H_Seed"],
        "efg":     ["Away eFG%", "Team1 eFG%", "Away eFG", "A_eFG",
                    "Home eFG%", "Team2 eFG%", "Home eFG", "H_eFG"],
        "tov":     ["Away TOV%", "Team1 TOV%", "Away TOV", "A_TOV",
                    "Home TOV%", "Team2 TOV%", "Home TOV", "H_TOV"],
        "orb":     ["Away ORB%", "Team1 ORB%", "Away ORB", "A_ORB",
                    "Home ORB%", "Team2 ORB%", "Home ORB", "H_ORB"],
        "ftr":     ["Away FTR", "Team1 FTR", "Away FT Rate", "A_FTR",
                    "Home FTR", "Team2 FTR", "Home FT Rate", "H_FTR"],
        "three_rate": ["Away 3P Rate", "Team1 3P Rate", "Away 3PA Rate", "A_3PR",
                       "Home 3P Rate", "Team2 3P Rate", "Home 3PA Rate", "H_3PR"],
    }

    mapping = {}
    for family, candidates in families.items():
        half = len(candidates) // 2
        away = next((c for c in candidates[:half] if c in df.columns), None)
        home = next((c for c in candidates[half:] if c in df.columns), None)
        mapping[family] = (away, home)
    return mapping

def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    out = normalize_cols(df).copy()
    side_cols = detect_side_columns(out)

    for family, (a_col, h_col) in side_cols.items():
        if a_col and h_col:
            out[f"diff_{family}"] = pd.to_numeric(out[a_col], errors="coerce") - pd.to_numeric(out[h_col], errors="coerce")

    # Keep legacy delta columns if they already exist
    legacy_pairs = {
        "delta OR": "diff_off_eff",
        "delta DR": "diff_def_eff",
        "delta AT": "diff_tempo",
    }
    for legacy, canonical in legacy_pairs.items():
        if legacy in out.columns and canonical not in out.columns:
            out[canonical] = pd.to_numeric(out[legacy], errors="coerce")

    # A few useful interactions if the base columns exist
    if {"diff_off_eff", "diff_def_eff"}.issubset(out.columns):
        out["net_strength_gap"] = out["diff_off_eff"] - out["diff_def_eff"]
    if {"diff_tempo", "diff_three_rate"}.issubset(out.columns):
        out["variance_gap"] = out["diff_tempo"].abs() + out["diff_three_rate"].abs()
    if "diff_seed" in out.columns:
        out["favorite_seed_edge"] = -1 * out["diff_seed"]  # smaller seed number is stronger

    return out

def choose_feature_columns(df: pd.DataFrame) -> list[str]:
    preferred = [
        "diff_off_eff", "diff_def_eff", "diff_tempo", "diff_seed",
        "diff_efg", "diff_tov", "diff_orb", "diff_ftr", "diff_three_rate",
        "net_strength_gap", "variance_gap", "favorite_seed_edge",
    ]
    available = [c for c in preferred if c in df.columns]
    if len(available) >= 3:
        return available

    # fallback: use existing numeric diff/delta columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    fallback = [
        c for c in numeric_cols
        if any(token in c.lower() for token in ["diff_", "delta", "seed", "adj", "or", "dr", "at"])
        and c not in {"Away Win", "Home Win", "Team1_Win", "Target", "target"}
    ]
    fallback = list(dict.fromkeys(fallback))
    if not fallback:
        raise ValueError("No usable numeric feature columns were found.")
    return fallback


In [13]:

# =========================
# 3. Load and prepare training data
# =========================
train_df = pd.read_csv(TRAIN_PATH)
train_df = normalize_cols(train_df)
train_df = train_df.dropna(how="all").copy()
train_df = add_engineered_features(train_df)

y, target_col = get_target_series(train_df)
feature_cols = choose_feature_columns(train_df)

model_df = train_df[feature_cols].copy()
model_df[target_col] = y
model_df = model_df.dropna(subset=[target_col]).copy()

X = model_df[feature_cols]
y = model_df[target_col].astype(int)

print(f"Training rows: {len(model_df):,}")
print(f"Target column: {target_col}")
print(f"Features used ({len(feature_cols)}): {feature_cols}")
model_df.head()


KeyError: "Could not find any of these columns: ['Away Win', 'AwayWin', 'away_win', 'Team1_Win', 'Team1 Win', 'team1_win', 'Target', 'target', 'WinnerFlag', 'winner_flag', 'Home Win', 'HomeWin', 'home_win']"


## Notes on the current feature strategy

This notebook deliberately prioritizes **difference features** because tournament prediction is a matchup problem.

Examples:
- `diff_off_eff` = Team1 offense - Team2 defense proxy
- `diff_seed` = Team1 seed - Team2 seed
- `variance_gap` = rough volatility indicator

That is a cleaner extension of the `delta OR`, `delta DR`, and `delta AT` logic you were already using.


In [ ]:

# =========================
# 4. Build candidate models
# =========================
numeric_features = feature_cols

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_features),
    ],
    remainder="drop",
)

log_reg = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=3000, C=1.0)),
])

rf = Pipeline(steps=[
    ("prep", ColumnTransformer(
        transformers=[
            ("num", Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
            ]), numeric_features)
        ],
        remainder="drop",
    )),
    ("model", RandomForestClassifier(
        n_estimators=600,
        max_depth=None,
        min_samples_leaf=3,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )),
])

hgb = Pipeline(steps=[
    ("prep", ColumnTransformer(
        transformers=[
            ("num", Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
            ]), numeric_features)
        ],
        remainder="drop",
    )),
    ("model", HistGradientBoostingClassifier(
        max_depth=5,
        learning_rate=0.03,
        max_iter=300,
        random_state=RANDOM_SEED,
    )),
])

models = {
    "log_reg": log_reg,
    "random_forest": rf,
    "hist_gradient_boosting": hgb,
}
list(models.keys())


In [ ]:

# =========================
# 5. Cross-validated evaluation
# =========================
def evaluate_models_cv(X: pd.DataFrame, y: pd.Series, models: dict[str, Pipeline], cv=CV) -> pd.DataFrame:
    rows = []
    for name, model in models.items():
        scores = cross_validate(
            model,
            X,
            y,
            cv=cv,
            scoring={
                "roc_auc": "roc_auc",
                "accuracy": "accuracy",
                "neg_log_loss": "neg_log_loss",
                "neg_brier": "neg_brier_score",
            },
            n_jobs=-1,
            error_score="raise",
        )
        rows.append({
            "model": name,
            "roc_auc_mean": np.mean(scores["test_roc_auc"]),
            "accuracy_mean": np.mean(scores["test_accuracy"]),
            "log_loss_mean": -np.mean(scores["test_neg_log_loss"]),
            "brier_mean": -np.mean(scores["test_neg_brier"]),
        })
    return pd.DataFrame(rows).sort_values(["log_loss_mean", "brier_mean", "roc_auc_mean"], ascending=[True, True, False])

cv_results = evaluate_models_cv(X, y, models)
cv_results


In [ ]:

# =========================
# 6. Fit calibrated final models + ensemble
# =========================
fitted_models = {}
oof_pred = pd.DataFrame(index=X.index)

for name, model in models.items():
    calibrated = CalibratedClassifierCV(estimator=clone(model), method="sigmoid", cv=5)
    pred = cross_val_predict(calibrated, X, y, cv=CV, method="predict_proba", n_jobs=-1)[:, 1]
    oof_pred[name] = pred
    calibrated.fit(X, y)
    fitted_models[name] = calibrated

oof_pred["ensemble_mean"] = oof_pred.mean(axis=1)

ensemble_metrics = {
    "roc_auc": roc_auc_score(y, oof_pred["ensemble_mean"]),
    "accuracy": accuracy_score(y, (oof_pred["ensemble_mean"] >= 0.5).astype(int)),
    "log_loss": log_loss(y, oof_pred["ensemble_mean"]),
    "brier": brier_score_loss(y, oof_pred["ensemble_mean"]),
}
ensemble_metrics


In [ ]:

# =========================
# 7. Diagnostics
# =========================
diag = pd.DataFrame({
    "actual": y,
    "ensemble_prob": oof_pred["ensemble_mean"],
}).copy()
diag["pred"] = (diag["ensemble_prob"] >= 0.5).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(diag.loc[diag["actual"] == 1, "ensemble_prob"], bins=25, alpha=0.7, label="Actual 1")
axes[0].hist(diag.loc[diag["actual"] == 0, "ensemble_prob"], bins=25, alpha=0.7, label="Actual 0")
axes[0].set_title("Ensemble predicted probability distribution")
axes[0].set_xlabel("Predicted probability Team1/Away wins")
axes[0].legend()

bins = np.linspace(0, 1, 11)
diag["bin"] = pd.cut(diag["ensemble_prob"], bins=bins, include_lowest=True)
calibration = diag.groupby("bin", observed=False).agg(
    mean_pred=("ensemble_prob", "mean"),
    observed=("actual", "mean"),
    count=("actual", "size"),
).dropna()

axes[1].plot([0, 1], [0, 1], linestyle="--")
axes[1].plot(calibration["mean_pred"], calibration["observed"], marker="o")
axes[1].set_title("Calibration curve")
axes[1].set_xlabel("Mean predicted probability")
axes[1].set_ylabel("Observed win rate")

plt.tight_layout()
plt.show()

calibration



## Optional: feature importance view

For interpretability, logistic regression coefficients are usually the cleanest place to start.
Tree models can be stronger, but the linear model often tells the easiest story.


In [ ]:

# =========================
# 8. Logistic coefficient view
# =========================
log_reg_fitted = fitted_models["log_reg"].estimator
lr_model = log_reg_fitted.named_steps["model"]

coef_df = pd.DataFrame({
    "feature": numeric_features,
    "coef": lr_model.coef_[0],
}).sort_values("coef", ascending=False)

plt.figure(figsize=(9, 6))
plt.barh(coef_df["feature"], coef_df["coef"])
plt.gca().invert_yaxis()
plt.title("Logistic regression coefficients")
plt.xlabel("Coefficient")
plt.show()

coef_df



# 9. Optional unsupervised team/game archetypes

This section is mainly exploratory.

Use it to identify clusters like:
- strong favorite / low-variance
- upset-friendly high-variance teams
- slow defensive grinders
- fragile favorites with turnover problems

It should support your bracket decisions, but not replace the probabilistic model.


In [ ]:

# =========================
# 9. Unsupervised exploration
# =========================
cluster_features = [c for c in feature_cols if c in X.columns][: min(8, len(feature_cols))]
cluster_df = X[cluster_features].copy()

cluster_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

X_cluster = cluster_pipe.fit_transform(cluster_df)

pca = PCA(n_components=2, random_state=RANDOM_SEED)
X_pca = pca.fit_transform(X_cluster)

kmeans = KMeans(n_clusters=4, n_init=25, random_state=RANDOM_SEED)
clusters = kmeans.fit_predict(X_cluster)

cluster_plot = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "cluster": clusters,
})

plt.figure(figsize=(8, 6))
plt.scatter(cluster_plot["PC1"], cluster_plot["PC2"], c=cluster_plot["cluster"])
plt.title("PCA + KMeans archetypes")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

cluster_summary = pd.DataFrame(X_cluster, columns=cluster_features).assign(cluster=clusters).groupby("cluster").mean()
cluster_summary



# 10. Functions for scoring new tournament matchups

This section turns the trained model into a bracket engine.

It needs:
1. a team stats file for the current season
2. a bracket matchup file


In [ ]:

# =========================
# 10. Matchup scoring helpers
# =========================
TEAM_NAME_CANDIDATES = ["Team", "team", "School", "TeamName", "TEAM"]

TEAM_STAT_ALIASES = {
    "off_eff": ["OR", "AdjO", "Adj OE", "AdjOE", "OffEff", "Adj_Off"],
    "def_eff": ["DR", "AdjD", "DefEff", "Adj_Def"],
    "tempo": ["AT", "AdjT", "Tempo", "Pace"],
    "seed": ["Seed", "seed"],
    "efg": ["eFG%", "eFG", "EFG"],
    "tov": ["TOV%", "TOV", "TO%"],
    "orb": ["ORB%", "ORB"],
    "ftr": ["FTR", "FT Rate"],
    "three_rate": ["3P Rate", "3PA Rate", "ThreeRate"],
}

def resolve_team_col(df: pd.DataFrame) -> str:
    return first_existing(df, TEAM_NAME_CANDIDATES, required=True)

def resolve_team_stat_columns(df: pd.DataFrame) -> dict[str, str]:
    cols = {}
    for family, aliases in TEAM_STAT_ALIASES.items():
        found = next((c for c in aliases if c in df.columns), None)
        if found:
            cols[family] = found
    if not cols:
        raise ValueError("No recognizable team stat columns were found in the team stats file.")
    return cols

def build_matchup_feature_frame(
    bracket_df: pd.DataFrame,
    team_stats_df: pd.DataFrame,
    required_features: list[str],
) -> pd.DataFrame:
    bracket_df = normalize_cols(bracket_df).copy()
    team_stats_df = normalize_cols(team_stats_df).copy()

    team_col = resolve_team_col(team_stats_df)
    stat_map = resolve_team_stat_columns(team_stats_df)
    stats = team_stats_df.set_index(team_col)

    team1_col = first_existing(bracket_df, ["Team1", "Away", "Away Team", "Visitor", "LowerSeedTeam"], required=True)
    team2_col = first_existing(bracket_df, ["Team2", "Home", "Home Team", "Host", "UpperSeedTeam"], required=True)

    rows = []
    missing_teams = set()

    for idx, row in bracket_df.iterrows():
        team1 = row[team1_col]
        team2 = row[team2_col]

        if team1 not in stats.index or team2 not in stats.index:
            missing_teams.update([t for t in [team1, team2] if t not in stats.index])
            rows.append({f: np.nan for f in required_features})
            continue

        rec = {}
        for family, col in stat_map.items():
            rec[f"diff_{family}"] = pd.to_numeric(stats.loc[team1, col], errors="coerce") - pd.to_numeric(stats.loc[team2, col], errors="coerce")

        if {"diff_off_eff", "diff_def_eff"}.issubset(rec):
            rec["net_strength_gap"] = rec["diff_off_eff"] - rec["diff_def_eff"]
        if {"diff_tempo", "diff_three_rate"}.issubset(rec):
            rec["variance_gap"] = abs(rec["diff_tempo"]) + abs(rec["diff_three_rate"])
        if "diff_seed" in rec:
            rec["favorite_seed_edge"] = -1 * rec["diff_seed"]

        rows.append({f: rec.get(f, np.nan) for f in required_features})

    feat_df = pd.DataFrame(rows, index=bracket_df.index)

    if missing_teams:
        print("Teams missing from team stats file:")
        print(sorted(missing_teams))

    return pd.concat([bracket_df, feat_df], axis=1)

def ensemble_predict_proba(df_features: pd.DataFrame, fitted_models: dict[str, CalibratedClassifierCV]) -> np.ndarray:
    preds = []
    for name, model in fitted_models.items():
        preds.append(model.predict_proba(df_features[feature_cols])[:, 1])
    return np.mean(np.column_stack(preds), axis=1)


In [ ]:

# =========================
# 11. Load current team stats + bracket
# =========================
team_stats = pd.read_csv(TEAM_STATS_PATH)
bracket = pd.read_csv(BRACKET_PATH)

bracket_scored = build_matchup_feature_frame(bracket, team_stats, feature_cols)
bracket_scored["Team1_Win_Prob"] = ensemble_predict_proba(bracket_scored, fitted_models)
bracket_scored["Team2_Win_Prob"] = 1 - bracket_scored["Team1_Win_Prob"]

display_cols = [c for c in ["Game_ID", "Round", "Region", "Slot", "Team1", "Team2", "Team1_Win_Prob", "Team2_Win_Prob"] if c in bracket_scored.columns]
bracket_scored[display_cols].head(20)



# 12. True advancing bracket engine

Yes — this is the actual point of the bracket model.

This upgraded section supports a **real bracket tree** where winners from earlier games automatically populate later-round matchups.

## Recommended bracket structure
Use a bracket file with these columns:
- `Game_ID` — unique ID for each game
- `Round` — numeric round (`1,2,3,4,5,6`)
- `Region` — optional
- `Slot` — optional display label
- `Team1` — actual team name for fixed teams (usually only Round 1)
- `Team2` — actual team name for fixed teams (usually only Round 1)
- `Team1_From` — prior `Game_ID` that supplies Team 1 for future rounds
- `Team2_From` — prior `Game_ID` that supplies Team 2 for future rounds

### Example
| Game_ID | Round | Slot | Team1 | Team2 | Team1_From | Team2_From |
|---|---:|---|---|---|---|---|
| 1 | 1 | R64-1 | Duke | Akron |  |  |
| 2 | 1 | R64-2 | Baylor | Mississippi State |  |  |
| 33 | 2 | R32-1 |  |  | 1 | 2 |

For first-round games, fill `Team1` and `Team2` directly.
For later rounds, leave `Team1`/`Team2` blank and point to earlier `Game_ID`s with `Team1_From` and `Team2_From`.


In [ ]:

# =========================
# 12. True advancing bracket engine helpers
# =========================

def _clean_game_id(x):
    if pd.isna(x):
        return None
    try:
        f = float(x)
        if f.is_integer():
            return str(int(f))
    except Exception:
        pass
    return str(x).strip()


def prepare_bracket_structure(bracket_df: pd.DataFrame) -> pd.DataFrame:
    df = normalize_cols(bracket_df).copy()

    required = ["Game_ID", "Round"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Bracket structure is missing required columns: {missing}")

    for col in ["Team1", "Team2", "Team1_From", "Team2_From", "Region", "Slot"]:
        if col not in df.columns:
            df[col] = np.nan

    df["Game_ID"] = df["Game_ID"].apply(_clean_game_id)
    df["Team1_From"] = df["Team1_From"].apply(_clean_game_id)
    df["Team2_From"] = df["Team2_From"].apply(_clean_game_id)
    df["Round"] = pd.to_numeric(df["Round"], errors="coerce")
    df = df.sort_values(["Round", "Game_ID"], kind="stable").reset_index(drop=True)

    if df["Game_ID"].duplicated().any():
        dupes = df.loc[df["Game_ID"].duplicated(), "Game_ID"].tolist()
        raise ValueError(f"Duplicate Game_ID values found: {dupes[:10]}")

    game_ids = set(df["Game_ID"])
    bad_refs = []
    for col in ["Team1_From", "Team2_From"]:
        refs = [x for x in df[col].dropna().tolist() if x not in game_ids]
        bad_refs.extend(refs)
    if bad_refs:
        raise ValueError(f"Some Team*_From references do not match any Game_ID: {sorted(set(bad_refs))[:10]}")

    return df


def score_matchup(team1: str, team2: str, team_stats_df: pd.DataFrame, required_features: list[str], fitted_models: dict):
    single = pd.DataFrame([{"Team1": team1, "Team2": team2}])
    feat = build_matchup_feature_frame(single, team_stats_df, required_features)
    p1 = float(ensemble_predict_proba(feat, fitted_models)[0])
    return feat.iloc[0].to_dict(), p1


def resolve_game_teams(row: pd.Series, winners: dict) -> tuple[str, str]:
    team1 = row.get("Team1")
    team2 = row.get("Team2")

    if pd.isna(team1) and pd.notna(row.get("Team1_From")):
        team1 = winners.get(row.get("Team1_From"))
    if pd.isna(team2) and pd.notna(row.get("Team2_From")):
        team2 = winners.get(row.get("Team2_From"))

    if pd.isna(team1) or pd.isna(team2):
        raise ValueError(
            f"Could not resolve teams for Game_ID={row.get('Game_ID')}. "
            f"Check Team1/Team2 and Team1_From/Team2_From wiring."
        )

    return str(team1), str(team2)


def simulate_bracket_tree_once(bracket_structure_df: pd.DataFrame, team_stats_df: pd.DataFrame, required_features: list[str], fitted_models: dict) -> pd.DataFrame:
    bracket = prepare_bracket_structure(bracket_structure_df)
    winners = {}
    rows = []

    for _, row in bracket.iterrows():
        team1, team2 = resolve_game_teams(row, winners)
        feat_dict, p1 = score_matchup(team1, team2, team_stats_df, required_features, fitted_models)
        winner = team1 if np.random.rand() < p1 else team2

        winners[row["Game_ID"]] = winner
        out = row.to_dict()
        out.update({
            "Resolved_Team1": team1,
            "Resolved_Team2": team2,
            "Team1_Win_Prob": p1,
            "Team2_Win_Prob": 1 - p1,
            "Winner": winner,
        })
        for k, v in feat_dict.items():
            if k not in out:
                out[k] = v
        rows.append(out)

    return pd.DataFrame(rows)


def build_bracket_pick_tree(bracket_structure_df: pd.DataFrame, team_stats_df: pd.DataFrame, required_features: list[str], fitted_models: dict, pick_mode: str = "max_prob") -> pd.DataFrame:
    bracket = prepare_bracket_structure(bracket_structure_df)
    winners = {}
    rows = []

    for _, row in bracket.iterrows():
        team1, team2 = resolve_game_teams(row, winners)
        feat_dict, p1 = score_matchup(team1, team2, team_stats_df, required_features, fitted_models)
        winner = team1 if p1 >= 0.5 else team2

        winners[row["Game_ID"]] = winner
        out = row.to_dict()
        out.update({
            "Resolved_Team1": team1,
            "Resolved_Team2": team2,
            "Team1_Win_Prob": p1,
            "Team2_Win_Prob": 1 - p1,
            "Picked_Winner": winner,
        })
        for k, v in feat_dict.items():
            if k not in out:
                out[k] = v
        rows.append(out)

    return pd.DataFrame(rows)


def summarize_bracket_tree_simulations(bracket_structure_df: pd.DataFrame, team_stats_df: pd.DataFrame, required_features: list[str], fitted_models: dict, n_sims: int = N_SIMS):
    round_counts = {}
    game_pick_counts = {}
    title_counts = {}

    for _ in range(n_sims):
        sim = simulate_bracket_tree_once(bracket_structure_df, team_stats_df, required_features, fitted_models)

        for _, row in sim.iterrows():
            rnd = int(row["Round"])
            winner = row["Winner"]
            round_counts[(winner, rnd)] = round_counts.get((winner, rnd), 0) + 1
            game_pick_counts[(row["Game_ID"], winner)] = game_pick_counts.get((row["Game_ID"], winner), 0) + 1

        final_round = sim["Round"].max()
        final_winner = sim.loc[sim["Round"] == final_round, "Winner"].iloc[-1]
        title_counts[final_winner] = title_counts.get(final_winner, 0) + 1

    round_rows = [
        {"Team": team, "Round": rnd, "Advance_Prob": cnt / n_sims}
        for (team, rnd), cnt in round_counts.items()
    ]
    round_summary = pd.DataFrame(round_rows).sort_values(["Round", "Advance_Prob"], ascending=[True, False]) if round_rows else pd.DataFrame(columns=["Team", "Round", "Advance_Prob"])

    game_rows = [
        {"Game_ID": gid, "Winner": team, "Pick_Prob": cnt / n_sims}
        for (gid, team), cnt in game_pick_counts.items()
    ]
    game_summary = pd.DataFrame(game_rows).sort_values(["Game_ID", "Pick_Prob"], ascending=[True, False]) if game_rows else pd.DataFrame(columns=["Game_ID", "Winner", "Pick_Prob"])

    title_rows = [
        {"Team": team, "Title_Prob": cnt / n_sims}
        for team, cnt in title_counts.items()
    ]
    title_summary = pd.DataFrame(title_rows).sort_values("Title_Prob", ascending=False) if title_rows else pd.DataFrame(columns=["Team", "Title_Prob"])

    return round_summary, game_summary, title_summary


In [ ]:

# =========================
# 13. Run advancing-bracket simulation + deterministic bracket picks
# =========================

bracket_structure = pd.read_csv(BRACKET_PATH)

# Deterministic pick sheet (always takes the higher win probability in each resolved game)
pick_tree = build_bracket_pick_tree(bracket_structure, team_stats, feature_cols, fitted_models)
pick_tree.to_csv(OUTPUT_DIR / "deterministic_bracket_picks.csv", index=False)

pick_display_cols = [
    c for c in [
        "Game_ID", "Round", "Region", "Slot",
        "Resolved_Team1", "Resolved_Team2",
        "Team1_Win_Prob", "Team2_Win_Prob", "Picked_Winner"
    ] if c in pick_tree.columns
]
pick_tree[pick_display_cols].head(20)

# Monte Carlo simulations with automatic advancement wiring
round_summary, game_summary, title_summary = summarize_bracket_tree_simulations(
    bracket_structure_df=bracket_structure,
    team_stats_df=team_stats,
    required_features=feature_cols,
    fitted_models=fitted_models,
    n_sims=N_SIMS,
)

round_summary.to_csv(OUTPUT_DIR / "team_advancement_probabilities.csv", index=False)
game_summary.to_csv(OUTPUT_DIR / "game_pick_probabilities.csv", index=False)
title_summary.to_csv(OUTPUT_DIR / "title_probabilities.csv", index=False)

print("Saved:")
print(" - deterministic_bracket_picks.csv")
print(" - team_advancement_probabilities.csv")
print(" - game_pick_probabilities.csv")
print(" - title_probabilities.csv")

print("
Top title odds:")
display(title_summary.head(15))


In [ ]:

# =========================
# 14. Suggested pick for each game
# =========================
if not game_summary.empty:
    best_pick_by_game = (
        game_summary
        .sort_values(["Game_ID", "Pick_Prob"], ascending=[True, False])
        .drop_duplicates("Game_ID")
        .reset_index(drop=True)
    )
    best_pick_by_game.to_csv(OUTPUT_DIR / "suggested_game_picks.csv", index=False)
    best_pick_by_game.head(30)
else:
    print("No game-level pick summary available.")



# 15. Optional game theory overlay using public pick percentages

If your bracket file includes:
- `PickPct1`
- `PickPct2`

this cell estimates a simple leverage view:

\[
EV \approx P(\text{win}) \times \text{round points} \times (1 - \text{public pick share})
\]

This is not a perfect pool model, but it is a very useful first-pass way to identify where to fade popular teams or back underpicked teams.


In [ ]:

# =========================
# 15. Expected value vs public picks
# =========================
public1 = next((c for c in ["PickPct1", "PublicPct1", "ESPNPickPct1"] if c in bracket_scored.columns), None)
public2 = next((c for c in ["PickPct2", "PublicPct2", "ESPNPickPct2"] if c in bracket_scored.columns), None)

if public1 and public2 and "Round" in bracket_scored.columns:
    ev_df = bracket_scored.copy()
    ev_df[public1] = pd.to_numeric(ev_df[public1], errors="coerce").fillna(0)
    ev_df[public2] = pd.to_numeric(ev_df[public2], errors="coerce").fillna(0)

    ev_df["RoundPoints"] = ev_df["Round"].map(ROUND_POINTS).fillna(0)
    ev_df["EV_Team1"] = ev_df["Team1_Win_Prob"] * ev_df["RoundPoints"] * (1 - ev_df[public1])
    ev_df["EV_Team2"] = ev_df["Team2_Win_Prob"] * ev_df["RoundPoints"] * (1 - ev_df[public2])

    ev_df["Recommended_Pick"] = np.where(ev_df["EV_Team1"] >= ev_df["EV_Team2"], ev_df["Team1"], ev_df["Team2"])
    ev_df["Recommended_EV"] = ev_df[["EV_Team1", "EV_Team2"]].max(axis=1)

    ev_cols = [c for c in ["Game_ID", "Round", "Team1", "Team2", "Team1_Win_Prob", "Team2_Win_Prob", public1, public2, "Recommended_Pick", "Recommended_EV"] if c in ev_df.columns]
    ev_df[ev_cols].to_csv(OUTPUT_DIR / "game_theory_recommendations.csv", index=False)
    ev_df[ev_cols].head(25)
else:
    print("Public pick percentage columns were not found, so the game theory overlay was skipped.")



# 16. Practical bracket rules

Use the model probabilities as the foundation, then sanity-check with a few tournament-specific rules:

1. Do not chase low-probability early-round upsets just because they are fun.
2. Give extra weight to late-round title equity.
3. Use leverage selectively when public pick data shows a favorite is over-owned.
4. Favor high-confidence teams for championship and Final Four slots.
5. Use volatility features for targeted early-round upset shots, not for the whole bracket.

A good bracket is usually **not** the bracket with the most upsets. It is the bracket with the best mix of:
- stable late-round picks
- a few high-value leverage spots
- calibrated win probabilities


In [ ]:

# =========================
# 17. Save key artifacts
# =========================
artifacts = {
    "feature_columns": feature_cols,
    "cv_results": cv_results.to_dict(orient="records"),
    "ensemble_metrics": ensemble_metrics,
    "round_points": ROUND_POINTS,
    "n_sims": N_SIMS,
}

with open(OUTPUT_DIR / "model_summary.json", "w", encoding="utf-8") as f:
    json.dump(artifacts, f, indent=2)

print(f"Saved outputs to: {OUTPUT_DIR.resolve()}")
print(sorted([p.name for p in OUTPUT_DIR.iterdir()]))



# 18. Next upgrades you can add later

The cleanest future improvements would be:

- season-based train/test splits instead of random splits
- injury flags and late-season momentum variables
- automatic import of public pick percentages
- bracket-tree propagation so later-round games populate automatically from prior simulated winners (now included)
- a pool-level optimization objective if you know your pool size and payout structure
- calibration comparison by seed matchup bucket, especially 11/6 and 13/4 games
